# Lab: Real 8-GPU Training Traces with HTA

**Goal:** read a *real* multi-rank distributed-training trace two ways — perceptually in Perfetto, and analytically with Meta's HolisticTraceAnalysis — and reconcile the two. The traces are HTA's demo set: an 8-rank vision-transformer training job, one Kineto JSON per rank.

**Protocol:** every exercise asks you to commit a number *from the timeline* before computing it. Write your guess down. The gap between your read and the computed answer is the thing being trained.

Companion: [torch.profiler & HTA tool page](https://sys-design-primer-cvw.pages.dev/trace-reading/tools/3-torch-profiler/).

In [ ]:
%pip install -q HolisticTraceAnalysis pandas
import os, urllib.request
TRACE_DIR = "vision_transformer"
os.makedirs(TRACE_DIR, exist_ok=True)
BASE = "https://raw.githubusercontent.com/facebookresearch/HolisticTraceAnalysis/main/tests/data/vision_transformer"
for r in range(8):
    f = f"rank-{r}.json.gz"
    if not os.path.exists(f"{TRACE_DIR}/{f}"):
        urllib.request.urlretrieve(f"{BASE}/{f}", f"{TRACE_DIR}/{f}")
print("downloaded:", sorted(os.listdir(TRACE_DIR)))

## Part 1 — Perfetto first (do NOT skip)

1. `gunzip -k vision_transformer/rank-0.json.gz` (Perfetto also accepts the .gz directly).
2. Open [ui.perfetto.dev](https://ui.perfetto.dev) → *Open trace file* → `rank-0.json`.
3. Find the `ProfilerStep` annotations; measure one step's wall time (W/A/S/D to zoom/pan, M to mark a span).

**Commit these numbers before Part 2** (edit this cell):
- Step time (ms): `____`
- Fraction of the step where the GPU compute stream is idle (eyeball): `____ %`
- Are NCCL kernels overlapped with compute, or serialized after backward? `____`
- The single longest-running kernel family you can find: `____`

In [ ]:
from hta.trace_analysis import TraceAnalysis
analyzer = TraceAnalysis(trace_dir=TRACE_DIR)

## Part 2 — Temporal breakdown: where does GPU time go?
Compute vs non-compute vs idle, per rank. Check your idle-fraction guess from Part 1 against rank 0's row. Then look *across* ranks: is the spread tight? A wide spread is the straggler signature.

In [ ]:
df = analyzer.get_temporal_breakdown(visualize=False)
df

## Part 3 — Communication–computation overlap
The single most interview-relevant number in a distributed trace: what fraction of communication time is hidden under compute. Compare with your Part-1 serialized-vs-overlapped call. Rule of thumb from the [pod-training answer](https://sys-design-primer-cvw.pages.dev/google-interview/6-answer-pod-training/): healthy DDP hides nearly all of it; an overlap % well below ~80 means exposed all-reduce tail.

In [ ]:
overlap = analyzer.get_comm_comp_overlap(visualize=False)
overlap

## Part 4 — Idle-time attribution
Idle is not one thing: HTA splits it into **host_wait** (CPU didn't launch work — launch-bound/input-bound), **kernel_wait** (waiting on another kernel/stream — serialization), and other. This is the notebook version of the nsys question "for each gap, what is running instead?"

In [ ]:
idle = analyzer.get_idle_time_breakdown(ranks=[0], visualize=False)
idle

## Part 5 — Kernel breakdown + straggler hunt
1. Does the top kernel match your Part-1 guess?
2. Using the temporal breakdown across ranks: which rank is slowest, and is its extra time in *compute* (data/work imbalance) or in *non-compute* (it's the one waiting — meaning the straggler is someone else)?
3. Write the five-sentence narration: step time → gap quantified → signature named → confirming measurement → fix with expected win.

In [ ]:
kdf = analyzer.get_gpu_kernel_breakdown(visualize=False, num_kernels=10)
kdf[0].head(10) if isinstance(kdf, tuple) else kdf

## Stretch
- `analyzer.get_cuda_kernel_launch_stats()` — launch-bound check: launch latency distribution.
- `analyzer.get_queue_length_summary()` — stream queue depth: is the CPU keeping the GPU fed?
- Re-run Parts 2–4 on a trace you captured yourself (Lab C of [hands-on profiling](https://sys-design-primer-cvw.pages.dev/trace-reading/3-hands-on-profiling/)) and diff against this healthy-ish baseline.

**Caveat for honesty:** this is one 8-GPU ViT job, not an LLM run — treat it as an anatomy specimen, not a performance bar. API names drift between HTA versions; if a call errors, check `dir(analyzer)`.